In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'sample_list'

In [3]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
69,sample_list,sample_name,character
70,sample_list,plot_name,character
71,sample_list,phenophase,character
72,sample_list,sample_fractional_cover,USER-DEFINED
73,sample_list,sample_fract_cover_method,USER-DEFINED
74,sample_list,understory,boolean
75,sample_list,fractional_cover_understory,numrange
76,sample_list,notes,character
77,sample_list,campaign_name,character
78,sample_list,sample_id,uuid


In [4]:
# view relevant data-types info
# for now we can't do this automatically by column_name, because column_name is not explicitly linked to enum_type in any way yet

data_types = data_types[data_types['Enum Type'].isin(['FRACTIONAL_class', 'FRACTIONAL_method'])]
data_types

,Schema,Enum Type,Enum Value
0,sbgplants,FRACTIONAL_class,char
1,sbgplants,FRACTIONAL_class,npv
2,sbgplants,FRACTIONAL_class,pv
3,sbgplants,FRACTIONAL_class,snow
4,sbgplants,FRACTIONAL_class,soil
5,sbgplants,FRACTIONAL_class,water
6,sbgplants,FRACTIONAL_method,Line-intercept-transect
7,sbgplants,FRACTIONAL_method,Point
8,sbgplants,FRACTIONAL_method,Quadrat


In [5]:
# load relevant output tables
campaign = pd.read_csv(os.path.join(out_folder, 'campaign.csv'))
plot_event_metadata = pd.read_csv(os.path.join(out_folder, 'plot_event_metadata.csv'))[['plot_name', 'fractional_cover_method']]

In [6]:
# load, update relevant raw tables

# fractional cover
fractional_cover = pd.read_csv(os.path.join(doi, 'fractional_cover.csv'))
# fix typos
fractional_cover.loc[fractional_cover.CoverCode=='engelmann', 'CoverCode'] = 'Engelmann'
fractional_cover.loc[fractional_cover.CoverCode=='RibMon', 'CoverCode'] = 'Gooseberry' # confirm this with Dana?
fractional_cover.loc[fractional_cover.CoverCode=='RubIda', 'CoverCode'] = 'Raspberry' # confirm this with Dana?
# merge duplicate rows
keys = ['CoverCode', 'SampleSiteCode']
agg_ = {
    'SamplingArea': 'first',
    'CollectionDate': 'first',
    'FractionalCover': 'sum',
    'Note': 'first'
}
fractional_cover = (
    fractional_cover
    .groupby(keys, dropna=False, sort=False)
    .agg(agg_)
    .reset_index()
)

# get species_or_type from spp list
species_list = pd.read_csv(os.path.join(doi, 'species_list.csv'))
species_list['species_or_type'] = species_list['Genus'] + ' ' + species_list['Species']
species_list.loc[species_list['species_or_type'].isna(), 'species_or_type'] = species_list.loc[species_list['Genus'].isna(), 'CoverCode']
species_list = species_list[['CoverCode', 'species_or_type']]

fractional_cover = pd.merge(fractional_cover, species_list, on='CoverCode', how='left', suffixes=('',''))
fractional_cover

,CoverCode,SampleSiteCode,SamplingArea,CollectionDate,FractionalCover,Note,species_or_type
0,Moss,276-ER18,RCK,6/23/2018,5,not in species list,Moss
1,PotPul,001-ER18,RM,6/14/2018,85,None,Potentilla pulcherrima
2,Litter,001-ER18,RM,6/14/2018,10,None,Litter
3,OF,001-ER18,RM,6/14/2018,5,None,OF
4,LupBak,002-ER18,RM,6/14/2018,30,None,Lupinus bakeri
...,...,...,...,...,...,...,...
1257,Lodgepole,474-ER18,GS,7/30/2018,100,None,Pinus contorta
1258,Lodgepole,475-ER18,GS,7/30/2018,100,None,Pinus contorta
1259,Lodgepole,476-ER18,GS,7/30/2018,100,None,Pinus contorta
1260,Engelmann,477-ER18,GS,7/30/2018,100,None,Picea engelmannii


In [7]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['plot_name'] = fractional_cover['SampleSiteCode']
out_table['species_or_type'] = fractional_cover['species_or_type']
out_table['notes'] = fractional_cover['Note']
out_table['campaign_name'] = campaign.campaign_name[0]

# making the assumption that fractional_cover_understory is fractional cover? Unclear, check
out_table['fractional_cover_understory'] = fractional_cover['FractionalCover']

# assign sample_name ? check approach
out_table['sample_name'] = out_table['plot_name'] + '_' + out_table['species_or_type']
out_table['sample_name'] = out_table['sample_name'].str.replace(' ', '', regex=False)

# map fractional class
fc_class = {
    'Bare': 'soil',
    'Litter': 'npv',
    # 'Moss': 'npv' # Dana said moss "might be pv"
}
out_table['sample_fractional_cover'] = out_table['species_or_type'].map(fc_class).fillna('pv')

# join fractional cover method
out_table = pd.merge(out_table, plot_event_metadata, on='plot_name', how='left', suffixes=('',''))
out_table['sample_fract_cover_method'] = out_table['fractional_cover_method']

# reorder columns
out_table = out_table[schema.column_name]

# populate sample_id uuid for thinking thorugh leaf_properties
out_table['sample_id'] = ['sample_'+str(x) for x in range(len(out_table))]

# unclear
# out_table['phenophase']
# out_table['understory']
# out_table['fractional_cover_understory']

# raster_plot_event_id??? If there are multiple raster_plot_event_ids per plot (one per fly over of each plot), which raster_plot_event_id would this be linked to? Or replciated per raster_plot_event_id? Perhaps only those within a given amount of time? Same question for plot_event_metadata

out_table

,sample_name,plot_name,phenophase,sample_fractional_cover,sample_fract_cover_method,understory,fractional_cover_understory,notes,campaign_name,sample_id,species_or_type,raster_plot_event_id
0,276-ER18_Moss,276-ER18,NaN,pv,Quadrat,NaN,5,not in species list,East River 2018,sample_0,Moss,NaN
1,001-ER18_Potentillapulcherrima,001-ER18,NaN,pv,Quadrat,NaN,85,None,East River 2018,sample_1,Potentilla pulcherrima,NaN
2,001-ER18_Litter,001-ER18,NaN,npv,Quadrat,NaN,10,None,East River 2018,sample_2,Litter,NaN
3,001-ER18_OF,001-ER18,NaN,pv,Quadrat,NaN,5,None,East River 2018,sample_3,OF,NaN
4,002-ER18_Lupinusbakeri,002-ER18,NaN,pv,Quadrat,NaN,30,None,East River 2018,sample_4,Lupinus bakeri,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1257,474-ER18_Pinuscontorta,474-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1257,Pinus contorta,NaN
1258,475-ER18_Pinuscontorta,475-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1258,Pinus contorta,NaN
1259,476-ER18_Pinuscontorta,476-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1259,Pinus contorta,NaN
1260,477-ER18_Piceaengelmannii,477-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1260,Picea engelmannii,NaN


In [8]:
# qaqc all plots sum to 100
tmp = fractional_cover.groupby('SampleSiteCode', as_index=False)['FractionalCover'].sum()
tmp.FractionalCover.unique()

array([100])

In [9]:
# confirm final column data types

print(out_table.dtypes)

# adjust as necessary
# out_table['understory'] = out_table['phenophase'].astype(bool)

out_table.dtypes

sample_name                    object
plot_name                      object
phenophase                     object
sample_fractional_cover        object
sample_fract_cover_method      object
understory                     object
fractional_cover_understory     int64
notes                          object
campaign_name                  object
sample_id                      object
species_or_type                object
raster_plot_event_id           object
dtype: object


sample_name                    object
plot_name                      object
phenophase                     object
sample_fractional_cover        object
sample_fract_cover_method      object
understory                     object
fractional_cover_understory     int64
notes                          object
campaign_name                  object
sample_id                      object
species_or_type                object
raster_plot_event_id           object
dtype: object

In [10]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)

In [11]:
out_table

,sample_name,plot_name,phenophase,sample_fractional_cover,sample_fract_cover_method,understory,fractional_cover_understory,notes,campaign_name,sample_id,species_or_type,raster_plot_event_id
0,276-ER18_Moss,276-ER18,NaN,pv,Quadrat,NaN,5,not in species list,East River 2018,sample_0,Moss,NaN
1,001-ER18_Potentillapulcherrima,001-ER18,NaN,pv,Quadrat,NaN,85,None,East River 2018,sample_1,Potentilla pulcherrima,NaN
2,001-ER18_Litter,001-ER18,NaN,npv,Quadrat,NaN,10,None,East River 2018,sample_2,Litter,NaN
3,001-ER18_OF,001-ER18,NaN,pv,Quadrat,NaN,5,None,East River 2018,sample_3,OF,NaN
4,002-ER18_Lupinusbakeri,002-ER18,NaN,pv,Quadrat,NaN,30,None,East River 2018,sample_4,Lupinus bakeri,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1257,474-ER18_Pinuscontorta,474-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1257,Pinus contorta,NaN
1258,475-ER18_Pinuscontorta,475-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1258,Pinus contorta,NaN
1259,476-ER18_Pinuscontorta,476-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1259,Pinus contorta,NaN
1260,477-ER18_Piceaengelmannii,477-ER18,NaN,pv,Visual,NaN,100,None,East River 2018,sample_1260,Picea engelmannii,NaN
